In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
model="gemini-2.5-flash",
temperature=0
)

In [ ]:
import sys
sys.path.append("..")

In [ ]:
from src.tools import read_calendar, get_customer_profile
tools = {
"read_calendar": read_calendar,
"get_customer_profile": get_customer_profile
}

In [ ]:
def triage_node(state):
    email = state["email"]

    prompt = f"""
You are an enterprise email triage assistant.

Classify the email into ONE of these labels:

respond:
- meeting reminders
- meeting schedules or changes
- calendar-related emails
- customer profile requests
- normal service queries

notify_human:
- invoices or payment dues
- billing or transaction issues
- fraud, security alerts
- login or password problems
- complaints or escalations

ignore:
- newsletters
- promotions
- greetings
- delivery or shipment updates
- internal reports
- general announcements

Email:
{email}

Return ONLY one word:
respond OR notify_human OR ignore
"""

    label = llm.invoke(prompt).content.strip().lower()
    return {**state, "triage": label}

In [ ]:
def react_agent(state):
    email = state["email"]

    prompt = f"""
You are an AI assistant for a financial services company.

Classify the email into exactly ONE of:
respond – customer is asking a question, requesting info, or needs a reply
ignore – spam, marketing, greetings, or no action needed
notify_human – complaints, legal threats, fraud, escalation, or sensitive issues

Rules:
• Complaints, angry tone, legal words, fraud, threats → notify_human
• Questions about loans, payments, documents → respond
• Thank you, promotions, newsletters → ignore

Return ONLY one word: respond, ignore, or notify_human.

Email:
{email}
"""

    response = llm.invoke(prompt).content

    if "TOOL:" in response:
        tool_name = response.replace("TOOL:", "").strip()
        tool_result = tools[tool_name]()
        return {**state, "response": tool_result}

    return {**state, "response": response}


In [ ]:
from langgraph.graph import StateGraph
graph = StateGraph(dict)
graph.add_node("triage", triage_node)
graph.add_node("react", react_agent)
def route(state):
 if state["triage"] == "respond":
    return "react"
 else:
    return "end"
graph.add_conditional_edges("triage", route)
graph.set_entry_point("triage")
app = graph.compile()

In [ ]:
import pandas as pd
emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("Tracing:", os.getenv("LANGCHAIN_TRACING_V2"))
print("Project:", os.getenv("LANGCHAIN_PROJECT"))
print("Key:", os.getenv("LANGCHAIN_API_KEY")[:8])


In [ ]:
from langsmith import Client
client = Client()
client.list_projects()

In [ ]:
list(client.list_projects())

In [ ]:
import time

results = []

for _, row in emails.head(10).iterrows():
    email = row["body"]

    output = app.invoke({"email": email})
    results.append({
        "email": email,
        "triage": output["triage"],
        "response": output.get("response", "")
    })

    time.sleep(15)  # important to avoid 429

In [ ]:
pd.DataFrame(results).to_csv("../data/milestone1_output.csv", index=False)

In [ ]:
import pandas as pd
gold = pd.read_csv("../data/golden_labels.csv")
pred = pd.read_csv("../data/milestone1_output.csv")


merged = gold.merge(
pred,
on="email",
how="inner",
suffixes=("_gold", "_pred")
)

accuracy = (merged["expected"] ==
merged["triage"]).mean()
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
prompt = f"""
You are an AI assistant for a financial services company.

Classify the email into exactly ONE of:
respond – customer is asking a question, requesting info, or needs a reply
ignore – spam, marketing, greetings, or no action needed
notify_human – complaints, legal threats, fraud, escalation, or sensitive issues

Rules:
• Complaints, angry tone, legal words, fraud, threats → notify_human
• Questions about loans, payments, documents → respond
• Thank you, promotions, newsletters → ignore

Return ONLY one word: respond, ignore, or notify_human.

Email:
{email}
"""

In [ ]:
"""What You Must Submit
Each intern must push:
data/milestone1_output.csv
data/golden_labels.csv
notebook/milestone1_final.ipynb
If any of you gets API quota error, you must wait and retry — Gemini is required as per project
document."""

In [ ]:
"""merged = gold.merge(
pred,
on="email",
how="inner",
suffixes=("_gold", "_pred")
)

accuracy = (merged["expected"] ==
merged["triage"]).mean()
accuracy"""